In [2]:
import sys
sys.path.insert(0,'/mnt/AEA8F340A8F3059D/sportsbet/ai-engine')

In [4]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from tools.mongodbtools import(
    fetchStockholdersByStock,
    fetchStockByPlayer,
    saveNotification,
    saveAiInsight
)
from config.settings import settings
import json,logging

In [5]:
from config.settings import settings
import json,logging

In [6]:
logger=logging.getLogger(__name__)

In [7]:
ALERT_PROMPT="""You are a sports alert system. Analyze the live match data and detect significant events.

LIVE MATCH DATA:
{match_data}

LIVE SCORE:
{live_score}

Detect events and generate alerts as JSON:
{{
  "events": [
    {{
      "type": "wicket|boundary|collapse|milestone|momentum_shift",
      "description": "what happened",
      "severity": "low|medium|high|critical",
      "affectedPlayers": ["player names involved"],
      "impact": "how this affects betting/stocks"
    }}
  ],
  "overallStatus": {{
    "matchSituation": "brief description of match state",
    "momentum": "team1|team2|even",
    "keyInsight": "most important takeaway"
  }},
  "alerts": [
    {{
      "title": "alert title",
      "message": "detailed alert message for user notification",
      "priority": "low|medium|high"
    }}
  ]
}}

If there are no significant events, return empty events array.
"""

In [12]:
async def alertGenerator(state):
    matchData=state.get("match-data",{})
    liveScore=state.get("live-score",{})
    if not matchData:
        return{
            "output":{
                "events":[],
                "alerts":[]
            }
        }
    llm=ChatGoogleGenerativeAI(
        model=settings.GEMINI_MODEL,
        google_api_key=settings.GEMINI_API_KEY,
        temperature=0.2
    )
    prompt=ChatPromptTemplate.from_messages([
        ("system",ALERT_PROMPT)
    ])
    try:
        result=await (prompt|llm).ainvoke({
            "match-data":json.dumps(matchData,default=str)[:1000],
            "live-score":json.dumps(liveScore,default=str)[:1000],
        })
        content=result.content.strip()
        if "```" in content:
            content=content.split("```")[1].split("```")[0]
            if content.startswith("json"):
                content=content[4:]
        output=json.loads(content)
    except Exception as e:
        logger.error(f"alert generation failed {e}")
        output={
            "events":[],
            "alerts":[]
        }
    alerts=output.get("alerts",[])
    for alert in alerts:
        if alert.get("priority") in ("high","medium"):
            playerIds=matchData.get("playerIds",[])
            for pid in playerIds[:5]:
                stock=await fetchStockByPlayer(str(pid))
                if stock:
                    holders=await fetchStockholdersByStock(str(stock["_id"]))
                    for holder in holders[:50]:
                        try:
                            await saveNotification(
                                str(holder.get("userId","")),
                                alert.get("message","Match alert"),
                                "alert"
                            )
                        except Exception:
                            pass
    
    matchId=str(matchData.get("_id",""))
    if matchId and alerts:
        try:
            from datetime import datetime,timedelta
            await saveAiInsight({
                "matchId":matchId,
                "type":"smart-alert",
                "title":alerts[0].get("title","Match Alert"),
                "content":output,
                "summary":output.get("overallStatus",{}).get("matchSituation",""),
                "score":0.8,
                "metadata":{
                    "alertCount":len(alerts),
                },
                "expiresAt":datetime.utcnow()+timedelta(hours=1)
            })
        except Exception:
            pass
    return{
        "output":output
    }